# Module 4 — Capsid assembly
### BruSafe / CAPSID · iGEM 2026

Explicit nucleation–elongation cascade over the **90 intermediate sizes** of the T=3 capsid. Unlike a three-state model, it follows every oligomer from 1 to 90 dimers.

This notebook produces three results that a three-state model cannot produce:

1. the nucleation lag period
2. the concentration-dependent **kinetic trap**
3. an **optimum** of the initial dimer concentration, which is the falsifiable prediction of M4

It is run with `Runtime → Run all`. The sweep takes about 1 min.

## 1 · Setup

In [ ]:
# ---------------------------------------------------------------
# Only numpy, scipy and matplotlib are used; all three are
# preinstalled in Colab.
# ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

RT    = 0.593    # kcal/mol at ~298 K
N_CAP = 90       # coat-protein dimers per T=3 capsid (Ni 1995)

## 2 · Parameters

In [ ]:
# Assembly parameters: (central value, (min, max), source)
PARAMS_M4 = {
    "k_a_uM_min":     (3.0,  (0.4, 4.0),  "Ceres 2002; Endres 2002"),
    "g_assoc_kcal":   (6.0,  (4.0, 8.0),  "Zlotnick 1994 (entropic penalty)"),
    "g_contact_kcal": (-4.0, (-5., -3.),  "Ceres 2002 (energy per contact)"),
    "n_c_nucleus":    (4,    (3, 6),      "Endres 2002 (nucleus size)"),
}

def central(params=PARAMS_M4):
    return {k: v[0] for k, v in params.items()}

q = central()
for k, (v, rng, src) in PARAMS_M4.items():
    print(f"{k:<16} {v:<7} range {str(rng):<13} {src}")

## 3 · The model

The **reverse** rate constants are not fitted freely: they follow from detailed balance and the contact energies. For this reason the model cannot reproduce the optimum by adjusting the parameters independently.

In [ ]:
def contacts_for_step(j_target, n_c):
    """Subunit-subunit contacts formed when a subunit is added
    to reach the j_target-mer.

    All of the physics of the model is in this function:

      pre-nucleus (j <= n_c):  1 contact  -> with the entropic penalty the
          step is net unfavourable. This is the nucleation barrier and the
          cause of the lag period.
      growth:                  2 contacts -> favourable, fast elongation.
      closure (last ~8):       3 contacts -> very favourable; closure of the
          vertices pulls strongly towards the complete capsid
          (Endres & Zlotnick 2002).
    """
    if j_target <= n_c:       return 1
    if j_target >= N_CAP - 8: return 3
    return 2


def build_rates(k_a_uM_min, g_assoc_kcal, g_contact_kcal, n_c_nucleus):
    """Forward and reverse rates.

    The forward rate k_a is the same for all steps. The reverse rates are
    not fitted freely: they follow from detailed balance and the contact
    energies, which prevents them from being adjusted independently.

      dG_add(j) = g_assoc + contacts(j) * g_contact
      Kd(j)     = exp(dG_add / RT)
      b_{j+1}   = k_a * Kd(j)
    """
    f = np.full(N_CAP, k_a_uM_min)
    b = np.zeros(N_CAP + 1)
    for j in range(1, N_CAP):
        c = contacts_for_step(j + 1, n_c_nucleus)
        dG_add = g_assoc_kcal + c * g_contact_kcal
        b[j + 1] = k_a_uM_min * np.exp(dG_add / RT)
    return f, b


def cascade_ode(t, x, f, b):
    """x[j-1] = concentration (uM) of the j-mer, j = 1..90.
    x[0] is the free coat-protein DIMER: the assembly subunit.
    Reaction j:  X_j + X_1 <-> X_{j+1}   (j=1 uses x_1^2)
    """
    x = np.clip(x, 0.0, None)
    x1 = x[0]
    J = np.zeros(N_CAP - 1)
    J[0] = f[0] * x1 * x1 - b[2] * x[1]
    for j in range(2, N_CAP):
        J[j - 1] = f[j - 1] * x[j - 1] * x1 - b[j + 1] * x[j]
    dx = np.zeros(N_CAP)
    dx[0] = -2 * J[0] - np.sum(J[1:])
    dx[1] = J[0] - J[1]
    for k in range(3, N_CAP):
        dx[k - 1] = J[k - 2] - J[k - 1]
    dx[N_CAP - 1] = J[N_CAP - 2]
    return dx


def simulate_cascade(D0_uM, k_a_uM_min, g_assoc_kcal, g_contact_kcal,
                     n_c_nucleus, t_max=240.0, nt=600):
    """Returns t, capsid(t), free(t), intermediates(t), full state.
    The fractions are mass fractions of the total subunit pool and sum to 1."""
    f, b = build_rates(k_a_uM_min, g_assoc_kcal, g_contact_kcal, n_c_nucleus)
    x0 = np.zeros(N_CAP); x0[0] = D0_uM
    t_eval = np.linspace(0, t_max, nt)
    sol = solve_ivp(cascade_ode, (0, t_max), x0, t_eval=t_eval, args=(f, b),
                    method="LSODA", rtol=1e-7, atol=1e-10)
    x = sol.y
    j = np.arange(1, N_CAP + 1)[:, None]
    capsid = N_CAP * x[-1] / D0_uM
    free   = x[0] / D0_uM
    inter  = (np.sum(j * x, axis=0) - x[0] - N_CAP * x[-1]) / D0_uM
    return t_eval, capsid, free, inter, x


# --- mass-conservation check: if this fails, the model is broken ---
t_c, cap_c, free_c, int_c, _ = simulate_cascade(2.0, **q)
err = np.max(np.abs(cap_c + free_c + int_c - 1.0))
print(f"maximum mass-conservation error: {err:.2e}  "
      f"({'OK' if err < 1e-6 else 'PROBLEM'})")

# --- free-energy profile per step ---
for j in [2, 4, 5, 30, 83, 90]:
    c = contacts_for_step(j, q["n_c_nucleus"])
    dG = q["g_assoc_kcal"] + c * q["g_contact_kcal"]
    print(f"  step -> {j:2d}-mer: {c} contact(s), dG_add = {dG:+.1f} kcal/mol, "
          f"Kd = {np.exp(dG/RT):9.2e} uM")

## 4 · The concentration sweep

This is the prediction that distinguishes this model from a simple diagram: **more dimer does not mean a better yield**. A random-assembly model predicts that the yield increases monotonically with concentration; this model predicts a collapse.

In [ ]:
# Concentration sweep that produces the falsifiable prediction of the model
D0_scan = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0])
yield_240 = []
inter_240 = []
for D0 in D0_scan:
    _, cap, free, inter, _ = simulate_cascade(D0, **q)
    yield_240.append(cap[-1])
    inter_240.append(inter[-1])
yield_240 = np.array(yield_240); inter_240 = np.array(inter_240)

i_opt = int(np.argmax(yield_240))
print("[D]0 (uM):", D0_scan)
print("capsid:   ", np.round(yield_240, 2))
print("trapped:  ", np.round(inter_240, 2))
print(f"\noptimum at [D]0 = {D0_scan[i_opt]} uM  ->  yield "
      f"{yield_240[i_opt]:.0%}")

## 5 · The figures

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.0, 4.1))

# ---- panel a: kinetics at three concentrations ----
for D0, c in [(1.0, "#17607a"), (2.0, "#0b6e5a"), (10.0, "#c0392b")]:
    t, cap, free, inter, _ = simulate_cascade(D0, **q)
    axA.plot(t, cap,   color=c, lw=2.0, label=f"[D]$_0$ = {D0:g} µM")
    axA.plot(t, inter, color=c, lw=1.2, ls="--", alpha=0.75)
axA.set_xlabel("time (min)")
axA.set_ylabel("subunit mass fraction")
axA.set_title("a   Capsid (solid) vs trapped intermediates (dashed)",
              loc="left")
axA.set_ylim(-0.03, 1.03)
axA.legend(frameon=False, fontsize=8)

# ---- panel b: the curve with an optimum ----
axB.plot(D0_scan, yield_240, "o-", color="#0b6e5a", lw=2.0, ms=6,
         label="complete capsid")
axB.plot(D0_scan, inter_240, "s--", color="#c0392b", lw=1.5, ms=5,
         alpha=0.8, label="trapped in intermediates")
axB.axvline(D0_scan[i_opt], color="#d4a017", lw=1.4, ls=":")
axB.annotate(f"optimum {D0_scan[i_opt]:g} µM\n({yield_240[i_opt]:.0%})",
             xy=(D0_scan[i_opt], yield_240[i_opt]),
             xytext=(D0_scan[i_opt] * 1.9, yield_240[i_opt] * 0.72),
             fontsize=8, color="#7a5c00",
             arrowprops=dict(arrowstyle="->", color="#d4a017", lw=1.0))
axB.set_xscale("log")
axB.set_xlabel(r"initial dimer [D]$_0$ (µM)")
axB.set_ylabel("fraction at 240 min")
axB.set_title("b   The falsifiable prediction: an optimum exists", loc="left")
axB.set_ylim(-0.03, 1.03)
axB.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig("M4_assembly.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved: M4_assembly.png")

## 6 · Composition of the trap

If the yield collapses, the mass must be somewhere. The following figure shows where, and gives the signature to look for by SEC-HPLC: a peak of intermediates of medium size and not of free monomer.

In [ ]:
# Composition of the kinetic trap: size distribution.
# Only the intermediates (1..89) are plotted; the complete capsid (j=90) is
# reported separately because, if included, it would compress the rest of the axis.
fig, ax = plt.subplots(figsize=(6.6, 4.1))
sizes = np.arange(1, N_CAP + 1)

for D0, c in [(2.0, "#0b6e5a"), (10.0, "#c0392b")]:
    _, _, _, _, x = simulate_cascade(D0, **q)
    massfrac = sizes * x[:, -1] / D0        # mass fraction per size
    inter = massfrac[:-1]                   # intermediates 1..89
    ax.plot(sizes[:-1], inter, color=c, lw=1.9,
            label=f"[D]$_0$ = {D0:g} µM  ·  capsid {massfrac[-1]:.0%}")
    mean_size = np.sum(sizes[:-1] * inter) / max(np.sum(inter), 1e-12)
    print(f"[D]0 = {D0:5g} uM -> mean intermediate size = {mean_size:5.1f} "
          f"dimers | trapped mass = {inter.sum():.2f} | "
          f"mass in capsid = {massfrac[-1]:.2f}")

ax.set_xlabel("intermediate size (dimers)")
ax.set_ylabel("mass fraction at 240 min")
ax.set_title("Where the mass accumulates when assembly stalls",
             loc="left")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig("M4_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 7 · Sensitivity of the optimum

Before proposing a specific sweep to the laboratory, it is useful to know how far the optimum moves if a literature parameter is wrong. This determines the grid of concentrations that must be covered.

In [ ]:
# Sensitivity: displacement of the optimum when a parameter is wrong
D0_fine = np.logspace(np.log10(0.3), np.log10(8.0), 26)

def optimum(**kw):
    pars = dict(q); pars.update(kw)
    ys = [simulate_cascade(d, **pars)[1][-1] for d in D0_fine]
    i = int(np.argmax(ys))
    return D0_fine[i], ys[i]

print(f"{'variant':<34}{'optimal [D]0':>13}{'yield':>13}")
base_D, base_y = optimum()
print(f"{'central (literature)':<34}{base_D:>10.2f} µM{base_y:>12.0%}")
print(f"\n[note] the COARSE grid of section 4 gave {D0_scan[i_opt]:g} µM "
      f"({yield_240[i_opt]:.0%}); this FINE grid finds {base_D:.2f} µM "
      f"({base_y:.0%}).")
print("The optimum is sharp: the yield drops ~20 points if the concentration deviates by a")
print("factor of 1.5, so the range between 1.5 and 3 uM should be sampled finely.\n")
for label, kw in [
    ("g_contact = -3 (weak contact)",    dict(g_contact_kcal=-3.0)),
    ("g_contact = -5 (strong contact)",  dict(g_contact_kcal=-5.0)),
    ("n_c = 3 (small nucleus)",          dict(n_c_nucleus=3)),
    ("n_c = 6 (large nucleus)",          dict(n_c_nucleus=6)),
    ("k_a = 0.4 (slow association)",     dict(k_a_uM_min=0.4)),
]:
    d, y = optimum(**kw)
    print(f"{label:<34}{d:>10.2f} µM{y:>12.0%}")

## Proposed experimental conditions

Sweep to be measured by SEC-HPLC:

| | |
|---|---|
| **[D]₀** | 0.5, 1, 1.5, 2, 2.5, 3, 5, 10 µM |
| **times** | 60, 120, 240 min |
| **replicates** | 3 per condition |
| **conditions** | 37 °C, 50 mM phosphate pH 7.2, 100 mM NaCl |
| **stopping the reaction** | transfer to 4 °C immediately |

The grid must cross the optimum and extend into the collapse region: if the measurements stop at 2 µM, the model cannot be falsified. Because the optimum is sharp (section 7), points every ~0.5 µM between 1.5 and 3 µM are required.

**Expected result:** maximum yield near 2 µM and, above 5 µM, mass in the form of intermediates of medium size and not of free monomer. If, instead, the yield increases monotonically up to 10 µM, the hypothesis of nucleation with a barrier is wrong and `contacts_for_step` must be revised.

## Coupling with M3

The input of M4 is the output of M3, that is, the dimer pool. With the central values, M3 predicts D_ss = 3.88 µM, above the optimum of 2 µM. Lowering the culture temperature brings the dimer closer to the optimum (see the M3 notebook), so **the induction temperature is a lever on assembly yield** and not only a growth parameter.